# Gold - Modelo Analítico, KPIs y Power BI

Objetivo de Gold: transformar los datasets curados de Silver en un modelo analítico listo para toma de decisiones y consumo en Power BI. Gold es la única capa que crea dimensiones, hechos, marts y KPIs.

Regla de arquitectura: Gold no lee CSV ni Raw. Todas las entradas analíticas vienen de `data/silver` en Parquet y las salidas se publican en `data/gold` también como Parquet Snappy.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name in {"bronze", "silver", "gold"}:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

spark = (
    SparkSession.builder
    .appName("municipal-medallion-profiling")
    .config("spark.sql.parquet.mergeSchema", "true")
    .getOrCreate()
)

def path_exists(path: str) -> bool:
    return Path(path).exists()

def read_parquet(path: str):
    if not path_exists(path):
        print(f"No existe: {path}")
        return None
    return spark.read.parquet(path)

def show_df(df, n=10, truncate=False):
    if df is None:
        print("DataFrame no disponible")
    else:
        df.show(n, truncate=truncate)

def count_nulls_and_blanks(df):
    exprs = []
    for c, dtype in df.dtypes:
        if dtype == "string":
            exprs.append(F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c))
        else:
            exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c))
    return df.select(exprs)

def summarize_table(name: str, df, business_keys=None):
    business_keys = business_keys or []
    rows = df.count()
    cols = len(df.columns)
    duplicates = rows - df.dropDuplicates().count()
    print(f"Tabla: {name}")
    print(f"Registros: {rows:,}")
    print(f"Columnas: {cols}")
    print(f"Duplicados exactos: {duplicates:,}")
    if business_keys and all(c in df.columns for c in business_keys):
        dup_keys = df.groupBy(*business_keys).count().filter("count > 1").count()
        print(f"Duplicados por clave {business_keys}: {dup_keys:,}")
    df.printSchema()
    return {"table": name, "rows": rows, "columns": cols, "duplicates": duplicates}

def domain_check(df, column, valid_values):
    return (
        df.groupBy(column)
        .count()
        .withColumn("is_valid_domain", F.col(column).isin(list(valid_values)))
        .orderBy(F.desc("count"))
    )

## 1. Lectura desde Silver Parquet

Objetivo: evidenciar que Gold parte de Silver, no de CSV. Este bloque inspecciona las tablas Silver que alimentan el modelo Gold.

In [ ]:
silver_root = PROJECT_ROOT / "data" / "silver"
gold_root = PROJECT_ROOT / "data" / "gold"

silver_inputs = {
    "municipalidades_curated": read_parquet(str(silver_root / "municipalidades_curated")),
    "renamu_curated": read_parquet(str(silver_root / "renamu_curated")),
    "ingresos_municipales_curated": read_parquet(str(silver_root / "ingresos_municipales_curated")),
    "predial_esat_curated": read_parquet(str(silver_root / "predial_esat_curated")),
    "sismepre_entidad_estado_curated": read_parquet(str(silver_root / "sismepre_entidad_estado_curated")),
    "sismepre_respuestas_curated": read_parquet(str(silver_root / "sismepre_respuestas_curated")),
    "categorias_municipalidades_curated": read_parquet(str(silver_root / "categorias_municipalidades_curated")),
}

inventory = []
for name, df in silver_inputs.items():
    path = silver_root / name
    inventory.append((name, path.exists(), len(list(path.rglob("*.parquet"))) if path.exists() else 0, df.count() if df is not None else 0))
spark.createDataFrame(inventory, ["silver_dataset", "path_exists", "parquet_files", "rows"]).show(50, truncate=False)

## 2. Salidas Gold disponibles

Objetivo: revisar dimensiones, facts, marts y tablas de KPIs ya persistidas. Si una salida falta, se debe ejecutar `python main_gold.py`.

In [ ]:
gold_tables = {
    "dim_municipalidad_gold": read_parquet(str(gold_root / "dim_municipalidad_gold")),
    "dim_ubigeo": read_parquet(str(gold_root / "dim_ubigeo")),
    "dim_tiempo": read_parquet(str(gold_root / "dim_tiempo")),
    "dim_clasificador_ingreso": read_parquet(str(gold_root / "dim_clasificador_ingreso")),
    "dim_estado_sismepre": read_parquet(str(gold_root / "dim_estado_sismepre")),
    "dim_formulario_sismepre": read_parquet(str(gold_root / "dim_formulario_sismepre")),
    "dim_pregunta_sismepre": read_parquet(str(gold_root / "dim_pregunta_sismepre")),
    "fact_ingresos_mensuales": read_parquet(str(gold_root / "fact_ingresos_mensuales")),
    "fact_ingresos_clasificador": read_parquet(str(gold_root / "fact_ingresos_clasificador")),
    "fact_predial_mensual": read_parquet(str(gold_root / "fact_predial_mensual")),
    "fact_sismepre_cumplimiento": read_parquet(str(gold_root / "fact_sismepre_cumplimiento")),
    "fact_sismepre_respuestas_resumen": read_parquet(str(gold_root / "fact_sismepre_respuestas_resumen")),
    "fact_renamu_gestion_tributaria": read_parquet(str(gold_root / "fact_renamu_gestion_tributaria")),
    "fact_renamu_software_at": read_parquet(str(gold_root / "fact_renamu_software_at")),
    "mart_dashboard_municipal": read_parquet(str(gold_root / "mart_dashboard_municipal")),
    "mart_kpi_resumen_ejecutivo": read_parquet(str(gold_root / "mart_kpi_resumen_ejecutivo")),
}

gold_inventory = []
for name, df in gold_tables.items():
    path = gold_root / name
    gold_inventory.append((name, path.exists(), len(list(path.rglob("*.parquet"))) if path.exists() else 0, df.count() if df is not None else 0))
spark.createDataFrame(gold_inventory, ["gold_table", "path_exists", "parquet_files", "rows"]).show(100, truncate=False)

## 3. KPIs de negocio

Gold publica aproximadamente cuatro KPIs ejecutivos:

1. **% ejecución de recaudación:** mide cuánto se recaudó respecto al PIM. Es clave para evaluar avance presupuestario municipal.
2. **Variación PIM - PIA:** mide cuánto cambió el presupuesto modificado frente al inicial. Ayuda a identificar ajustes presupuestarios relevantes.
3. **Efectividad predial:** mide recaudación predial frente a emisión predial. Ayuda a decidir dónde reforzar cobranza.
4. **Capacidad tecnológica tributaria:** mide qué porcentaje de herramientas tributarias RENAMU tiene la municipalidad: SRTM, software propio de rentas y catastro.

In [ ]:
kpi = gold_tables.get("mart_kpi_resumen_ejecutivo")
if kpi is not None:
    kpi.select(
        "year", "SEC_EJEC", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE",
        "categoria_municipalidad", "recaudacion_total", "pim_total",
        "kpi_pct_ejecucion_recaudacion", "kpi_variacion_pim_pia",
        "recaudacion_predial_total", "emision_predial_total",
        "kpi_efectividad_predial", "kpi_capacidad_software_pct",
        "prioridad_intervencion"
    ).orderBy(F.desc("recaudacion_total")).show(20, truncate=False)

### KPI 1: % ejecución de recaudación

Qué mide: `recaudacion_total / pim_total * 100`. Relevancia: muestra cuánto de lo programado/modificado realmente ingresó.

In [ ]:
if kpi is not None:
    kpi.groupBy("year").agg(
        F.sum("recaudacion_total").alias("recaudacion_total"),
        F.sum("pim_total").alias("pim_total"),
        (F.sum("recaudacion_total") / F.sum("pim_total") * 100).alias("pct_ejecucion_promedio")
    ).orderBy("year").show(30, truncate=False)

### KPI 2: Variación presupuestaria PIM - PIA

Qué mide: diferencia entre presupuesto modificado e inicial. Relevancia: identifica municipalidades con ampliaciones o reducciones fuertes.

In [ ]:
if kpi is not None:
    kpi.select("year", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE", "kpi_variacion_pim_pia") \
       .orderBy(F.desc("kpi_variacion_pim_pia")) \
       .show(20, truncate=False)

### KPI 3: Efectividad predial

Qué mide: `recaudacion_predial_total / emision_predial_total * 100`. Relevancia: ayuda a priorizar asistencia en cobranza predial.

In [ ]:
if kpi is not None:
    kpi.filter(F.col("kpi_efectividad_predial").isNotNull()) \
       .groupBy("DEPARTAMENTO_NOMBRE", "categoria_municipalidad") \
       .agg(F.avg("kpi_efectividad_predial").alias("efectividad_predial_promedio")) \
       .orderBy(F.desc("efectividad_predial_promedio")) \
       .show(40, truncate=False)

### KPI 4: Capacidad tecnológica tributaria

Qué mide: porcentaje de tres herramientas RENAMU disponibles: SRTM, software de rentas y catastro. Relevancia: aproxima capacidad operativa para administrar tributos.

In [ ]:
if kpi is not None:
    kpi.groupBy("categoria_municipalidad").agg(
        F.count("*").alias("municipalidades_periodo"),
        F.avg("kpi_capacidad_software_pct").alias("capacidad_software_promedio")
    ).orderBy("categoria_municipalidad").show(30, truncate=False)

## 4. Validaciones de calidad Gold

Objetivo: asegurar que las tablas analíticas no pierden claves y que los KPIs respetan reglas de negocio, especialmente división por cero y categorías A-G.

In [ ]:
validations = []
for name, df in gold_tables.items():
    if df is None:
        validations.append((name, "availability", "failed", "No existe tabla Gold"))
        continue
    validations.append((name, "availability", "passed", f"{df.count():,} filas"))
    if "SEC_EJEC" in df.columns:
        null_sec = df.filter(F.col("SEC_EJEC").isNull()).count()
        validations.append((name, "sec_ejec_not_null", "passed" if null_sec == 0 else "failed", str(null_sec)))
    if "categoria_municipalidad" in df.columns:
        invalid_cat = df.filter(F.col("categoria_municipalidad").isNotNull() & ~F.col("categoria_municipalidad").isin(list("ABCDEFG"))).count()
        validations.append((name, "categoria_a_g", "passed" if invalid_cat == 0 else "failed", str(invalid_cat)))

if kpi is not None:
    invalid_exec = kpi.filter((F.col("pim_total") == 0) & F.col("kpi_pct_ejecucion_recaudacion").isNotNull()).count()
    validations.append(("mart_kpi_resumen_ejecutivo", "pim_zero_execution_null", "passed" if invalid_exec == 0 else "failed", str(invalid_exec)))

spark.createDataFrame(validations, ["table", "check", "status", "detail"]).show(200, truncate=False)

## 5. Tablas agregadas para Power BI

Objetivo: mapear las salidas Gold a las seis páginas de Power BI. Las tablas `pbi_dashboard_01..06` se publican como carpetas Parquet bajo `data/gold` para conectarlas directamente desde Power BI Desktop, sin Hive ni ODBC.

In [ ]:
dashboard_mapping = [
    ("pbi_dashboard_01", "Recaudación Municipal vs Capacidad Tributaria", "SIAF + RENAMU + Categorías"),
    ("pbi_dashboard_02", "Recaudación por Clasificador de Ingreso", "SIAF + Clasificador + Categorías"),
    ("pbi_dashboard_03", "Predial vs Efectividad", "SISMEPRE + Categorías"),
    ("pbi_dashboard_04", "Distribución de Efectividad Predial", "SISMEPRE + Categorías"),
    ("pbi_dashboard_05", "Software Tributario Municipal", "RENAMU + Categorías"),
    ("pbi_dashboard_06", "Priorización de Municipalidades", "SIAF + SISMEPRE + RENAMU + Categorías"),
    ("pbi_kpi_resumen_ejecutivo", "Resumen ejecutivo de KPIs", "Mart Gold de KPIs"),
]
spark.createDataFrame(dashboard_mapping, ["tabla_gold_powerbi", "pagina", "fuentes"]).show(truncate=False)

## 6. Ejemplos visuales rápidos

Objetivo: generar tablas de apoyo que se pueden convertir en barras, tarjetas o tablas en Power BI.

In [ ]:
if kpi is not None:
    print("Top departamentos por recaudación")
    kpi.groupBy("year", "DEPARTAMENTO_NOMBRE") \
       .agg(F.sum("recaudacion_total").alias("recaudacion_total")) \
       .orderBy(F.desc("recaudacion_total")) \
       .show(20, truncate=False)

    print("Prioridad de intervención")
    kpi.groupBy("year", "prioridad_intervencion") \
       .count() \
       .orderBy("year", "prioridad_intervencion") \
       .show(50, truncate=False)

## 7. Persistencia final en Parquet

Objetivo: confirmar que Gold quedó físicamente en Parquet Snappy. La escritura oficial la hace `main_gold.py` mediante `GoldStorage`. Este bloque verifica la presencia de archivos y deja un ejemplo seguro de escritura opcional.

In [ ]:
for table in ["dim_municipalidad_gold", "fact_ingresos_mensuales", "fact_predial_mensual", "mart_kpi_resumen_ejecutivo"]:
    path = gold_root / table
    print(table, "parquet_files=", len(list(path.rglob("*.parquet"))) if path.exists() else 0, "path=", path)

WRITE_NOTEBOOK_PREVIEW = False
if WRITE_NOTEBOOK_PREVIEW and kpi is not None:
    preview_path = gold_root / "_notebook_validation" / "mart_kpi_resumen_ejecutivo_preview"
    kpi.write.mode("overwrite").option("compression", "snappy").partitionBy("year").parquet(str(preview_path))
    print(f"Preview escrito en {preview_path}")

## 8. Revisión de consumo en Power BI

Para la entrega final se omite Hive. Power BI debe consumir directamente los Parquet Gold:

- `data/gold/pbi_dashboard_01`
- `data/gold/pbi_dashboard_02`
- `data/gold/pbi_dashboard_03`
- `data/gold/pbi_dashboard_04`
- `data/gold/pbi_dashboard_05`
- `data/gold/pbi_dashboard_06`

Esta decisión elimina los errores ODBC/HiveServer2 y mantiene la arquitectura Medallion clara: Bronze, Silver y Gold almacenan Parquet; Power BI consume Gold.

In [ ]:
powerbi_tables = [f"pbi_dashboard_{i:02d}" for i in range(1, 7)]
powerbi_inventory = []
for table in powerbi_tables:
    path = gold_root / table
    powerbi_inventory.append((table, str(path), path.exists(), len(list(path.rglob("*.parquet"))) if path.exists() else 0))

spark.createDataFrame(
    powerbi_inventory,
    ["tabla_gold_powerbi", "parquet_path", "exists", "parquet_files"],
).show(truncate=False)

## 9. Conclusión Gold

Gold consume Silver Parquet, construye constelación de hechos/dimensiones, genera marts para seis dashboards y publica KPIs ejecutivos. El consumo final de Power BI se hace directamente desde Parquet Gold, no desde Excel ni Hive.